In [9]:
%pip install pdfminer.six

# Part 1: Mount Google Drive & Import Libraries

In [10]:
from google.colab import drive
drive.mount('/content/drive')

import os
import re
import logging
from pdfminer.high_level import extract_text
from io import BytesIO

# Configuration: Define paths in Google Drive
BASE_DIR = '/content/drive/MyDrive'
PATH_PDF = os.path.join(BASE_DIR, 'Penalaran Komputer (460, 464)', 'Data', 'HAM')
PATH_OUTPUT = os.path.join(BASE_DIR, 'Penalaran Komputer (460, 464)', 'Data', 'raw')
LOG_DIR = os.path.join(BASE_DIR, 'Penalaran Komputer (460, 464)', 'logs')
LOG_PATH = os.path.join(LOG_DIR, 'cleaning.log')

# Ensure directories exist
for path in [PATH_PDF, PATH_OUTPUT, LOG_DIR]:
    os.makedirs(path, exist_ok=True)

# Initialize logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(LOG_PATH, mode='w', encoding='utf-8'),
    ],
    force=True
)

logging.info("Logging initialized at %s", LOG_PATH)
print(f"PDF folder   : {PATH_PDF}")
print(f"Output folder: {PATH_OUTPUT}")
print(f"Log file     : {LOG_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PDF folder   : /content/drive/MyDrive/Penalaran Komputer (460, 464)/Data/HAM
Output folder: /content/drive/MyDrive/Penalaran Komputer (460, 464)/Data/raw
Log file     : /content/drive/MyDrive/Penalaran Komputer (460, 464)/logs/cleaning.log


# Part 2: Utility Functions for PDF Processing

In [11]:
def extract_pdf_text(pdf_path):
    try:
        with open(pdf_path, 'rb') as file:
            text = extract_text(BytesIO(file.read()))
        logging.info(f"Extracted text from {os.path.basename(pdf_path)}")
        return text
    except Exception as e:
        logging.error(f"Failed to extract text from {pdf_path}: {e}")
        return ""

def clean_text(text):
    if not text:
        logging.info("No text to clean")
        return ""
    text = text.replace("M a h ka m a h A g u n g R e p u blik In d o n esia\n", "")
    text = text.replace("Disclaimer\n", "")
    text = text.replace(
        "Kepaniteraan Mahkamah Agung Republik Indonesia berusaha untuk selalu mencantumkan informasi paling kini dan akurat sebagai bentuk komitmen Mahkamah Agung untuk pelayanan publik, transparansi dan akuntabilitas\n", ""
    )
    text = text.replace(
        "pelaksanaan fungsi peradilan. Namun dalam hal-hal tertentu masih dimungkinkan terjadi permasalahan teknis terkait dengan akurasi dan keterkinian informasi yang kami sajikan, hal mana akan terus kami perbaiki dari waktu kewaktu.\n", ""
    )
    text = text.replace(
        "Dalam hal Anda menemukan inakurasi informasi yang termuat pada situs ini atau informasi yang seharusnya ada, namun belum tersedia, maka harap segera hubungi Kepaniteraan Mahkamah Agung RI melalui :\n", ""
    )
    text = text.replace(
        "Email : kepaniteraan@mahkamahagung.go.id    Telp : 021-384 3348 (ext.318)\n", ""
    )
    text = text.replace("Direktori Putusan Mahkamah Agung Republik Indonesia", "")
    text = text.replace("putusan.mahkamahagung.go.id", "")
    text = text.replace("Pid.I.A.3", "")
    text = text.replace("Mahkamah Agung Republik Indonesia", "")
    text = re.sub(r'Halaman \d+', '', text)
    text = text.lower()
    text = re.sub(r'[^a-z0-9]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    original_length = len(re.sub(r'[^a-z0-9]', '', text.replace(" ", "")))
    cleaned_length = len(text.replace(" ", ""))
    integrity = (cleaned_length / original_length) * 100 if original_length > 0 else 0
    if integrity < 80:
        logging.warning(f"Text integrity below 80% for document: {integrity:.1f}%")
        return ""
    logging.info(f"Cleaned text successfully, integrity: {integrity:.1f}%")
    return text

def save_text_file(text, index, path):
    if not text:
        logging.info("No text to save")
        return None
    file_name = f"case_{index:03d}.txt"
    save_path = os.path.join(path, file_name)
    with open(save_path, 'w', encoding='utf-8') as f:
        f.write(text)
    logging.info(f"Saved cleaned text: {file_name}")
    return file_name

# Part 3: Main Processing Function

In [12]:
def process_pdfs(max_files=50):
    logging.info("Starting PDF processing with max_files=%d", max_files)

    # Clear existing files in output directory
    for file in os.listdir(PATH_OUTPUT):
        file_path = os.path.join(PATH_OUTPUT, file)
        if os.path.isfile(file_path):
            os.remove(file_path)
            logging.info(f"Deleted existing file: {file_path}")
    logging.info("Cleared all existing files in %s", PATH_OUTPUT)

    pdf_files_list = [f for f in os.listdir(PATH_PDF) if f.endswith('.pdf')]
    logging.info(f"Found {len(pdf_files_list)} PDF files to process")

    if not pdf_files_list:
        logging.warning(f"No PDF files found in {PATH_PDF}")
        print("No PDF files found.")
        return

    processed_files_text = 0
    for index, pdf_file in enumerate(pdf_files_list, start=1):
        if processed_files_text >= max_files:
            logging.info("Max file limit reached, stopping processing")
            break

        pdf_path = os.path.join(PATH_PDF, pdf_file)
        logging.info(f"Started processing PDF: {pdf_file}")
        text = extract_pdf_text(pdf_path)
        cleaned_text = clean_text(text)
        if cleaned_text:
            saved_file = save_text_file(cleaned_text, index, PATH_OUTPUT)
            if saved_file:
                processed_files_text += 1
        else:
            logging.info(f"No valid text after cleaning for {pdf_file}")

    final_count = len([f for f in os.listdir(PATH_OUTPUT) if f.startswith('case_') and f.endswith('.txt')])
    logging.info(f"Processing complete. Generated {final_count} text files.")
    print(f"Processing completed. Generated {final_count} text files.")

# Part 4: Run the Processor

In [13]:
process_pdfs(max_files=50)

Processing completed. Generated 50 text files.
